## Evaluate integration models

In [ ]:
# !bash -c 'eval "$(conda shell.bash hook)" \
#     && conda activate /work/islet_cartography_scrna/scrna_cartography_py_analysis \
#     && python -m ipykernel install --user \
#        --name scrna_cartography_py_analysis \
#        --display-name "py_analysis"'

## Load Libraries

In [ ]:
import os
import sys
import glob
from pyhere import here

import anndata
import numpy as np
import pandas as pd
import scanpy as sc
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
import seaborn as sns
import torch
import skmisc

import pickle

# My modules / functions
sys.path.append(str(here('scripts/misc')))  
import my_anndata as ma

## Set up parameters

In [ ]:
# Directories
ma.create_directories(dir_path = str(here('data/integration/first_pass')))
ma.create_directories(dir_path = str(here('data/integration/first_pass/tmp')))
ma.create_directories(dir_path = str(here('data/integration/first_pass/embedding')))
ma.create_directories(dir_path = str(here('data/integration/first_pass/files')))
ma.create_directories(dir_path = str(here('data/integration/first_pass/plot')))
ma.create_directories(dir_path = str(here('data/integration/first_pass/objects')))
ma.create_directories(dir_path = str(here('data/integration/first_pass/models')))

In [ ]:
# Saving
base_dir = str(here('data/integration/first_pass/'))
file_dir = os.path.join(base_dir, 'files') 
plot_dir = os.path.join(base_dir, 'plots') 
tmp_dir = os.path.join(base_dir, 'tmp') 
emb_dir = os.path.join(base_dir, 'embedding') 
objects_dir = os.path.join(base_dir, 'objects') 

In [ ]:
hvgs = [500, 1000, 2000]
methods = ["unintegrated", "scvi", "sysvi", "scpoli"]
adata_dir = tmp_dir  # directory with preprocessed HVG files
file_dir = os.path.join(base_dir, 'files')  # directory to save embeddings
model_dir = os.path.join(base_dir, 'models')   # directory to save models
key_batch = ["technical_integration", "ic_id_donor_integrate"]   # adjust to your obs keys
latent_dims_list = [15, 20, 30]
embedding_dims_list = [5, 10, 20]
label_key = 'study_cell_annotation_harmonized' 

In [ ]:
# plotting
sc.set_figure_params(figsize=(5, 5), frameon=False)
# save path
sc.settings.figdir = plot_dir
%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

## Load data

In [ ]:
adata =sc.read_h5ad(here('data/anndata/AB_combined.h5ad'))

In [ ]:
# Add technical variation
adata.obs['technical_integration'] = (
    adata.obs['cell_nuclei'].astype(str) + "_" + adata.obs['count_quantification'].astype(str)
)

# copy of raw counts
adata.raw = adata.copy()

# make na unknown, otherwise it will not work
adata.obs[label_key] = adata.obs[label_key].fillna('unknown')

In [ ]:
# subset for test
# number per group
n_per_group = 3000 // adata.obs['name'].nunique()

# sample indices
sampled_obs = (
    adata.obs
    .groupby('name', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), n_per_group), random_state=0))
)

# subset AnnData
adata= adata[sampled_obs.index, :].copy()

In [ ]:
for n in hvgs:
    
    ad = adata.copy()
    sc.pp.highly_variable_genes(
        ad,
        n_top_genes=n,
        flavor="seurat_v3",
        layer="counts",
        batch_key =  key_batch[0],
        subset=True)

    # Extract hvgs
    hvg_list = ad.var[ad.var['highly_variable']].index.tolist()
    
    # Save hvgs 
    hvg_path = os.path.join(file_dir, f'hvgs_{n}.txt')
    
    with open(hvg_path, 'w') as f:
        for gene in hvg_list:
            f.write(gene + '\n')
            
    # Save adata object
    adata_file = os.path.join(objects_dir, f"adata_{n}_hvg.h5ad")
    ad.write(adata_file)
    
    # Save adata object for scpoli
    adata_file_scpoli = os.path.join(adata_dir, f"adata_scpoli_{n}_hvg.h5ad")
    del ad.uns["log1p"]
    ad.write(adata_file_scpoli)

    for latent_dims in latent_dims_list:
        for embedding_dims in embedding_dims_list:
            key_save = f"{'_'.join(key_batch)}_latent{latent_dims}_embed{embedding_dims}"

        # Run unintegrated (PCA)
        subprocess.run([
            "conda", "run", "-n", "py_analysis", "python", "run_no_int.py",
            str(n), adata_file, model_dir, emb_dir, key_save, str(latent_dims)
        ])
        
    
        # Run scVI
        subprocess.run([
            "conda", "run", "-n", "py_analysis", "python", "run_scvi.py",
            str(n), adata_file, model_dir, emb_dir, key_batch[0], key_batch[1], key_save,  str(latent_dims), str(embedding_dims)
        ])
        
        # Run SysVI
        subprocess.run([
            "conda", "run", "-n", "py_analysis", "python", "run_sysvi.py",
            str(n), adata_file, model_dir, emb_dir, key_batch[0], key_batch[1], key_save, str(latent_dims), str(embedding_dims)
        ])
        
        # Run scPoli
        subprocess.run([
            "conda", "run", "-n", "scpoli", "python", "run_scpoli.py",
            str(n), adata_file_scpoli, model_dir, emb_dir, key_batch[1], key_save, str(latent_dims), str(embedding_dims)
        ])

## Add latent embedding

In [ ]:
for method in methods:
    for n in hvgs:
        for latent_dims in latent_dims_list:
            for embedding_dims in embedding_dims_list:
                key_save = f"{'_'.join(key_batch)}_latent{latent_dims}_embed{embedding_dims}"
            
                df_path = os.path.join(emb_dir, f"{method}_latent_embd_{n}_{key_save}.csv")
                
                if not os.path.exists(df_path):
                    print(f' Missing embedding: {df_path}')
                    continue
                
                latent_df = pd.read_csv(df_path, index_col=0)
                latent_df = latent_df.loc[adata.obs_names]  # ensure correct cell order
                adata.obsm[f"{method}_{n}_{key_save}"] = latent_df.to_numpy()

## Generate UMAP

In [ ]:
latent_keys = [key for key in adata.obsm.keys() if any(m in key for m in methods)]

for key in latent_keys:
    print(f"Computing neighbors and UMAP for {key}...")
    
    # Compute neighbors on the latent space
    sc.pp.neighbors(adata, use_rep=key, key_added=f"{key}_neighbors")
    
    # Compute UMAP, store in obsm
    sc.tl.umap(adata, neighbors_key=f"{key}_neighbors", key_added=f"{key}_umap")
    
    # Optional: plot
    sc.pl.umap(adata, color="study_cell_annotation_harmonized", 
               use_rep=f"{key}_umap", 
               title=f"UMAP {key}", 
               save=os.path.join("figures", f"_UMAP_{key}.png"))